# Generate predictions

get predict elasticity tensor, derived properties and save to json

In [1]:
import torch
import pandas as pd
from pymatgen.core.structure import Structure
from pymatgen.analysis.elasticity import ElasticTensor
from matten.predict import predict
from matten.utils import CartesianTensorWrapper


/home/qygao/.conda/envs/matten/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/qygao/.conda/envs/matten/lib/python3.10/site-packages/torchmetrics/utilities/imports.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [8]:
## SETTINGS

# data split
# SPLIT = 'train'
# SPLIT = 'val'
SPLIT = 'test'


MODEL_IDX = '7ws4jdfi/checkpoints/'
# MODEL_IDX = 'model_2'
# MODEL_IDX = 'model_3'
# MODEL_IDX = 'model_4'
# MODEL_IDX = 'model_5'

MODEL = f'/home/qygao/matten/scripts/wandb_logs/matten/{MODEL_IDX}'


## get target and predictions

In [4]:

filename = (
    "/home/qygao/matten"
    f"/datasets/crystal_elasticity_tensor_{SPLIT}.json"
)
df = pd.read_json(filename)

print(len(df))
print(df.columns)
df.head()

1027
Index(['structure', 'formula_pretty', 'crystal_system', 'elastic_tensor',
       'elastic_tensor_voigt', 'split'],
      dtype='object')


,structure,formula_pretty,crystal_system,elastic_tensor,elastic_tensor_voigt,split
5601,"{'@module': 'pymatgen.core.structure', '@class...",SrCuO2,orthorhombic,"[[[[188.92258398, 0.0, 0.0], [0.0, 94.6799928,...","[[188.92258398, 94.6799928, 25.63916148, 0.0, ...",train
2108,"{'@module': 'pymatgen.core.structure', '@class...",Ti3Au,cubic,"[[[[168.02043234, 0.0, 0.0], [0.0, 117.4076542...","[[168.02043234, 117.4076542, 117.4076542, 0.0,...",train
2516,"{'@module': 'pymatgen.core.structure', '@class...",Ti2TcRh,cubic,"[[[[380.44360583, 0.0, 0.0], [0.0, 114.9664217...","[[380.44360583, 114.96642177, 114.96642177, 0....",train
7318,"{'@module': 'pymatgen.core.structure', '@class...",ErGa5Co,tetragonal,"[[[[167.45943092, 0.0, 0.0], [0.0, 39.46690416...","[[167.45943092, 39.46690416, 49.76305771, 0.0,...",train
33,"{'@module': 'pymatgen.core.structure', '@class...",MoPt3,cubic,"[[[[354.81570936, 0.0, 0.0], [0.0, 201.0811404...","[[354.81570936, 201.08114046, 201.08114046, 0....",train


In [6]:
targets = df["elastic_tensor"].tolist()
targets = [torch.tensor(t) for t in targets]

In [9]:
structures = df["structure"].apply(lambda x: Structure.from_dict(x)).tolist()

predictions_tensor = predict(structures, model_identifier=MODEL)

predictions = [torch.tensor(p) for p in predictions_tensor]

/home/qygao/.conda/envs/matten/lib/python3.10/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


FileNotFoundError: [Errno 2] No such file or directory: '/home/qygao/matten/scripts/wandb_logs/matten/7ws4jdfi/checkpoints/config_final.yaml'

### MAEs

In [70]:
assert len(predictions) == len(df)

In [71]:
def get_spherical_tensor(t: list[torch.Tensor]):
    """Convert a Cartesian tensor to a spherical tensor."""
    t = torch.stack(t)
    converter = CartesianTensorWrapper("ijkl=jikl=klij")
    spherical = converter.from_cartesian(t)
    return spherical


def mae(a: torch.Tensor, b: torch.Tensor):
    return torch.mean(torch.abs(a - b))


In [72]:
p = torch.stack(predictions)
t = torch.stack(targets)

### MAE in cartesian space

In [73]:
mae(p,t)

tensor(3.1237)

In [74]:
### MAE in spherical space

In [75]:
ps = get_spherical_tensor(predictions)
ts = get_spherical_tensor(targets)
mae(ps,ts)

tensor(6.5212)

### Select properties

In [76]:

df_new = pd.DataFrame({
    'structure': df['structure'],
    'crystal_system': df['crystal_system'],
    'elastic_tensor': df['elastic_tensor_full'],
    'elastic_tensor_voigt': df['elastic_tensor_voigt'],
    'elastic_tensor_pred': [p.tolist() for p in predictions_tensor],
    'elastic_tensor_voigt_pred': [p.voigt.tolist() for p in predictions_tensor],

})

df_new.head()

,structure,crystal_system,elastic_tensor,elastic_tensor_voigt,elastic_tensor_pred,elastic_tensor_voigt_pred
9735,"{'@module': 'pymatgen.core.structure', '@class...",cubic,"[[[[494.67250903, 0.0, 0.0], [0.0, 117.2927166...","[[494.67250903, 117.29271667, 117.29271667, 0....","[[[[376.4710693359375, -3.038149714029714e-07,...","[[376.4710693359375, 161.23448181152344, 161.2..."
5358,"{'@module': 'pymatgen.core.structure', '@class...",cubic,"[[[[241.10189638, 0.0, 0.0], [0.0, 111.4459865...","[[241.10189638, 111.44598653, 111.44598653, 0....","[[[[228.95770263671875, 3.325562147438177e-06,...","[[228.95770263671875, 117.85057830810547, 117...."
6087,"{'@module': 'pymatgen.core.structure', '@class...",cubic,"[[[[94.08028962, 0.0, 0.0], [0.0, 30.40224479,...","[[94.08028962, 30.40224479, 30.40224479, 0.0, ...","[[[[76.64810943603516, 6.394893148353731e-08, ...","[[76.64810943603516, 12.941889762878418, 12.94..."
7154,"{'@module': 'pymatgen.core.structure', '@class...",cubic,"[[[[58.54093163, 0.0, 0.0], [0.0, 40.04168995,...","[[58.54093163, 40.04168995, 40.04168995, 0.0, ...","[[[[42.979393005371094, -7.539387070210068e-07...","[[42.979393005371094, 39.02738952636719, 39.02..."
2595,"{'@module': 'pymatgen.core.structure', '@class...",cubic,"[[[[75.24083529, 0.0, 0.0], [0.0, 38.40263146,...","[[75.24083529, 38.40263146, 38.40263146, 0.0, ...","[[[[70.86753845214844, -3.312551143608289e-06,...","[[70.86753845214844, 41.16896438598633, 41.168..."


In [77]:
def get_property(t: ElasticTensor, prop: str='k_voigt'):
    try:
         return getattr(t, prop)
    except:
        return None


expected = ['k_voigt', 'k_reuss', 'k_vrh', 'g_voigt', 'g_reuss', 'g_vrh', 'y_mod']

for mode in ['', '_pred']:
    tensors = df_new['elastic_tensor'+mode].map(lambda x: ElasticTensor(x))

    for k in expected:
        df_new[k+mode] =  tensors.apply(lambda x: get_property(x, k))


# remove failed
for mode in ['', '_pred']:
    for k in expected:
        df_new = df_new[df_new[k+mode].notna()]

In [78]:
## convert y_mod to GPa
df_new['y_mod'] =df_new['y_mod'] / 1e9
df_new['y_mod_pred'] =df_new['y_mod_pred'] / 1e9

In [79]:
print('Number of final data points', len(df_new))
df_new.head()


Number of final data points 1021


,structure,crystal_system,elastic_tensor,elastic_tensor_voigt,elastic_tensor_pred,elastic_tensor_voigt_pred,k_voigt,k_reuss,k_vrh,g_voigt,g_reuss,g_vrh,y_mod,k_voigt_pred,k_reuss_pred,k_vrh_pred,g_voigt_pred,g_reuss_pred,g_vrh_pred,y_mod_pred
9735,"{'@module': 'pymatgen.core.structure', '@class...",cubic,"[[[[494.67250903, 0.0, 0.0], [0.0, 117.2927166...","[[494.67250903, 117.29271667, 117.29271667, 0....","[[[[376.4710693359375, -3.038149714029714e-07,...","[[376.4710693359375, 161.23448181152344, 161.2...",243.085981,243.085981,243.085981,128.541051,112.309060,120.425055,310.071856,232.980011,232.980011,232.980011,90.344142,88.274149,89.309146,237.571084
5358,"{'@module': 'pymatgen.core.structure', '@class...",cubic,"[[[[241.10189638, 0.0, 0.0], [0.0, 111.4459865...","[[241.10189638, 111.44598653, 111.44598653, 0....","[[[[228.95770263671875, 3.325562147438177e-06,...","[[228.95770263671875, 117.85057830810547, 117....",154.664623,154.664623,154.664623,68.342608,68.220008,68.281308,178.566141,154.886292,154.886292,154.886292,64.542906,63.667597,64.105251,169.000160
6087,"{'@module': 'pymatgen.core.structure', '@class...",cubic,"[[[[94.08028962, 0.0, 0.0], [0.0, 30.40224479,...","[[94.08028962, 30.40224479, 30.40224479, 0.0, ...","[[[[76.64810943603516, 6.394893148353731e-08, ...","[[76.64810943603516, 12.941889762878418, 12.94...",51.628260,51.628260,51.628260,38.662994,37.809853,38.236423,91.997770,34.177294,34.177294,34.177294,28.635438,28.403101,28.519269,66.938771
7154,"{'@module': 'pymatgen.core.structure', '@class...",cubic,"[[[[58.54093163, 0.0, 0.0], [0.0, 40.04168995,...","[[58.54093163, 40.04168995, 40.04168995, 0.0, ...","[[[[42.979393005371094, -7.539387070210068e-07...","[[42.979393005371094, 39.02738952636719, 39.02...",46.208104,46.208104,46.208104,37.662878,18.571903,28.117391,70.127998,40.344724,40.344724,40.344724,21.254646,4.545028,12.899837,34.972172
2595,"{'@module': 'pymatgen.core.structure', '@class...",cubic,"[[[[75.24083529, 0.0, 0.0], [0.0, 38.40263146,...","[[75.24083529, 38.40263146, 38.40263146, 0.0, ...","[[[[70.86753845214844, -3.312551143608289e-06,...","[[70.86753845214844, 41.16896438598633, 41.168...",50.682033,50.682033,50.682033,26.100225,24.429310,25.264767,64.994482,51.068489,51.068489,51.068489,20.934230,19.628588,20.281409,53.731258


## Save to json

In [80]:
df_new.to_json(f'./results/elastic_tensor_{MODEL_IDX}_{SPLIT}.json')